# XClinVision: Data Exploration Notebook
**Day 1: EDA & Data Preparation**

This notebook explores the chest X-ray dataset structure, visualizes samples,
and analyzes class distributions for the XClinVision project.

## 1. Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path().absolute().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

## 2. Dataset Paths Configuration

In [ ]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print(f"Data directory: {DATA_DIR}")
print(f"Raw data exists: {RAW_DIR.exists()}")
print(f"Processed data exists: {PROCESSED_DIR.exists()}")

## 3. Load Dataset Metadata

In [ ]:
def load_dataset_metadata(data_dir: Path) -> pd.DataFrame:
    """Load and merge metadata from all dataset sources."""
    
    records = []
    
    # Check for available datasets
    datasets = {
        "chest_xray_pneumonia": RAW_DIR / "chest_xray_pneumonia",
        "tb_chest_xray": RAW_DIR / "tb_chest_xray",
    }
    
    for dataset_name, dataset_path in datasets.items():
        if dataset_path.exists():
            print(f"Found dataset: {dataset_name}")
            
            # Scan directory structure
            for split in ["train", "val", "test"]:
                split_path = dataset_path / split
                if split_path.exists():
                    for class_name in ["NORMAL", "PNEUMONIA", "Tuberculosis", "Normal"]:
                        class_path = split_path / class_name
                        if class_path.exists():
                            for img_file in class_path.glob("*.jpeg"):
                                records.append({
                                    "dataset": dataset_name,
                                    "split": split,
                                    "class": class_name,
                                    "filepath": str(img_file),
                                    "filename": img_file.name,
                                })
    
    return pd.DataFrame(records)

# Load metadata
df = load_dataset_metadata(RAW_DIR)
print(f"\nTotal images: {len(df)}")
df.head()

## 4. Class Distribution Analysis

In [ ]:
# Overall class distribution
class_counts = df["class"].value_counts()
print("Class Distribution:")
print(class_counts)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0])
axes[0].set_title("Class Distribution (Count)")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v, str(v), ha="center", va="bottom")

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct="%1.1f%%")
axes[1].set_title("Class Distribution (Percentage)")

plt.tight_layout()
plt.show()

## 5. Sample Image Visualization

In [ ]:
def visualize_samples(df: pd.DataFrame, n_samples: int = 3):
    """Visualize sample images from each class."""
    
    classes = df["class"].unique()
    n_classes = len(classes)
    
    fig, axes = plt.subplots(n_classes, n_samples, figsize=(n_samples * 4, n_classes * 4))
    
    for i, class_name in enumerate(classes):
        class_df = df[df["class"] == class_name]
        samples = class_df.sample(n=min(n_samples, len(class_df)))
        
        for j, (_, row) in enumerate(samples.iterrows()):
            img = Image.open(row["filepath"])
            if n_classes == 1:
                ax = axes[j]
            else:
                ax = axes[i, j]
            ax.imshow(img, cmap="gray")
            ax.set_title(f"{class_name}\n{row['filename'][:30]}...")
            ax.axis("off")
    
    plt.tight_layout()
    plt.show()

# Show samples
if len(df) > 0:
    visualize_samples(df, n_samples=3)
else:
    print("No images found. Please download datasets first.")

## 6. Summary Statistics

In [ ]:
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total images: {len(df)}")
print(f"Number of classes: {df['class'].nunique() if len(df) > 0 else 0}")
print(f"Classes: {df['class'].unique().tolist() if len(df) > 0 else 'N/A'}")
print("\nClass balance:")
if len(df) > 0:
    for cls, count in df["class"].value_counts().items():
        pct = count / len(df) * 100
        print(f"  {cls}: {count} ({pct:.1f}%)")
print("=" * 50)